# Customer Churn Prediction — Phase 1: Data Ingestion

**Business Goal:**  
Telecom companies lose ~2–3% of subscribers each month to churn.  
With ~7,000 customers and an average monthly charge of ~\$65, even a **5% churn reduction**  
saves approximately **\$22,750/month** in recurring revenue.  

This notebook loads the IBM Telco Churn dataset, inspects data quality,  
and fixes the one known data type issue before any analysis begins.

---

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Display settings — show all columns, clean float formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

DATA_PATH = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'
print('Libraries loaded.')

Libraries loaded.


## 1. Load the Dataset

In [2]:
df = pd.read_csv(DATA_PATH)

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
df.head()

Shape: 7,043 rows × 21 columns
Memory usage: 6984.7 KB


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Schema Inspection — Data Types & Nulls

**What we're looking for:**
- Are all column types correct? (e.g., numbers stored as strings?)
- Any missing values?
- What does the target variable (`Churn`) look like?

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
# Summary statistics — numerical columns only
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.00,7043.00,7043.00
mean,0.16,32.37,64.76
std,0.37,24.56,30.09
min,0.00,0.00,18.25
25%,0.00,9.00,35.50
50%,0.00,29.00,70.35
75%,0.00,55.00,89.85
max,1.00,72.00,118.75


### 🔍 Spot the Bug: TotalCharges is type `object` (string), not `float`

**Why does this happen?**  
Some rows have `TotalCharges = " "` (a blank space string) instead of a number.  
Pandas sees even one non-numeric value and keeps the entire column as `object`.  
The blank rows belong to new customers with **tenure = 0** — they've never been billed yet.

**Fix strategy:**  
1. Replace blank strings with `NaN`  
2. Cast the column to `float`  
3. Fill `NaN` with `0.0` — because a customer with 0 months tenure has R\$0 total charges (factually correct, not an imputation guess)

In [5]:
# Step 1: Find the problematic rows
blank_mask = df['TotalCharges'].str.strip() == ''
print(f'Rows with blank TotalCharges: {blank_mask.sum()}')
print()
print('Their tenure values (new customers):')
print(df.loc[blank_mask, ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']])

Rows with blank TotalCharges: 11

Their tenure values (new customers):
      customerID  tenure  MonthlyCharges TotalCharges
488   4472-LVYGI       0           52.55             
753   3115-CZMZD       0           20.25             
936   5709-LVOEQ       0           80.85             
1082  4367-NUYAO       0           25.75             
1340  1371-DWPAZ       0           56.05             
3331  7644-OMVMY       0           19.85             
3826  3213-VVOLG       0           25.35             
4380  2520-SGTTA       0           20.00             
5218  2923-ARZLG       0           19.70             
6670  4075-WKNIU       0           73.35             
6754  2775-SEFEE       0           61.90             


In [6]:
# Step 2: Fix the column
# pd.to_numeric with errors='coerce' turns non-parseable strings into NaN automatically
# WHY coerce not raise? Because we EXPECT some blanks — we want NaN, not a crash.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Step 3: Fill the 11 NaNs with 0 (tenure=0 means no charges yet)
df['TotalCharges'] = df['TotalCharges'].fillna(0.0)

print('TotalCharges dtype after fix:', df['TotalCharges'].dtype)
print('Remaining nulls in TotalCharges:', df['TotalCharges'].isna().sum())

TotalCharges dtype after fix: float64
Remaining nulls in TotalCharges: 0


## 3. Full Null Check Across All Columns

In [7]:
null_summary = df.isnull().sum()
null_summary = null_summary[null_summary > 0]

if len(null_summary) == 0:
    print('No null values anywhere in the dataset.')
else:
    print('Columns with nulls:')
    print(null_summary)

No null values anywhere in the dataset.


## 4. Target Variable Preview

In [8]:
churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100

summary = pd.DataFrame({
    'Count'      : churn_counts,
    'Percentage' : churn_pct.round(2)
})
print('Churn Distribution:')
print(summary)
print()
print('Class imbalance ratio (No:Yes) =',
      f"{churn_counts['No'] / churn_counts['Yes']:.1f}:1")

Churn Distribution:
       Count  Percentage
Churn                   
No      5174       73.46
Yes     1869       26.54

Class imbalance ratio (No:Yes) = 2.8:1


## 5. Quick Duplicate Check

In [9]:
dupes = df.duplicated().sum()
unique_customers = df['customerID'].nunique()

print(f'Duplicate rows       : {dupes}')
print(f'Unique customer IDs  : {unique_customers:,}')
print(f'Total rows           : {len(df):,}')
print()
if dupes == 0 and unique_customers == len(df):
    print('Each row is a unique customer. Dataset is clean.')

Duplicate rows       : 0
Unique customer IDs  : 7,043
Total rows           : 7,043

Each row is a unique customer. Dataset is clean.


## 6. Save Clean Version
Save the type-fixed dataframe so Phase 2 doesn't need to repeat the fix.

In [10]:
PROCESSED_PATH = '../data/processed/telco_churn_clean.csv'
df.to_csv(PROCESSED_PATH, index=False)
print(f'Saved to: {PROCESSED_PATH}')
print(f'Shape   : {df.shape}')
print()
print('Phase 1 Complete. Proceed to 02_eda.ipynb')

Saved to: ../data/processed/telco_churn_clean.csv
Shape   : (7043, 21)

Phase 1 Complete. Proceed to 02_eda.ipynb
